
# Evolet TMH extraction — optimized local single-LLM notebook

tuned for speed on the 147 TMH PDFs.

## What changed
- **Lossless first**: page ledger and note ledger are saved before any modeling
- **Triage pipeline**: deterministic regex extraction handles easy notes first
- **LLM rescue only**: the local model is used only on unresolved, clinically dense notes
- **No lossy final rewrite**: final records are merged in Python, not re-summarized by another model
- **Fast defaults**: native PDF text first, OCR only for weak pages, exact-duplicate note skipping, batched generation

## Default model
- `Qwen/Qwen2.5-1.5B-Instruct`

## Easy fallback
- `HuggingFaceTB/SmolLM2-1.7B-Instruct`


## Installs the working stack: pymupdf, pillow, python-doctr, transformers, accelerate, bitsandbytes, huggingface_hub, pandas, numpy, tqdm, orjson, rapidfuzz, python-dateutil, pydantic, and json-repair.

In [ ]:
# Cell 1 — install minimal packages
import sys
print(sys.version)
print(sys.executable)

!{sys.executable} -m pip install -U \
  pymupdf pillow python-doctr \
  transformers accelerate bitsandbytes safetensors huggingface_hub \
  pandas numpy tqdm orjson rapidfuzz python-dateutil pydantic json-repair


3.10.20 (main, Mar 23 2026, 03:02:05) [GCC 15.2.0]
/home/pardeep/venvs/evolet310/bin/python


## This opening block uses a practical single-notebook stack: PyMuPDF for PDF reading, Doctr for OCR, Hugging Face Transformers for the LLM, and lightweight data tools for ledgers and QC. That is a sensible choice because the pipeline needs to run on a VM/Jupyter workflow without requiring a big service architecture. The config is centralized early so the same notebook can switch between pilot and full runs safely. The helper layer is especially important because later cells depend on consistent JSON writing, logging, and recovery behavior. Better alternatives could be a packaged project with YAML configs, Hydra, Pydantic Settings, and a CLI runner, but this notebook approach was likely chosen because it is faster to debug, easier to rerun cell by cell, and better for iterative GPU work on medical PDFs.

In [ ]:
# Cell 2 — config, paths, runtime knobs

from pathlib import Path
import os
import torch

CANDIDATE_INPUTS = [
    Path("/home/pardeep/data/TMH_Patient_Reports"),
    Path("/content/TMH_Patient_Reports"),
    Path("/content/drive/MyDrive/Colab Notebooks/TMH_Patient_Reports"),
]
INPUT_PDF_DIR = next((p for p in CANDIDATE_INPUTS if p.exists()), CANDIDATE_INPUTS[0])

BASE_OUT_ROOT = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized")
if not str(BASE_OUT_ROOT).startswith("/home/pardeep") and Path("/content").exists():
    BASE_OUT_ROOT = Path("/content/Evolet_Qwen15B_Optimized")

# The pilot is already stable, so default to the full 147-PDF run.
# You can still force the pilot with:
#   export EVOLET_RUN_MODE=pilot10
RUN_MODE = os.environ.get("EVOLET_RUN_MODE", "full147").strip().lower()
if RUN_MODE not in {"pilot10", "full147"}:
    RUN_MODE = "full147"

PILOT_PDF_LIMIT = 10
LIMIT_PDFS = PILOT_PDF_LIMIT if RUN_MODE == "pilot10" else None
RUN_TAG = f"pilot_{PILOT_PDF_LIMIT:03d}" if LIMIT_PDFS is not None else "full_147"
OUT_ROOT = BASE_OUT_ROOT / RUN_TAG

PAGE_LEDGER_DIR = OUT_ROOT / "page_ledger"
NOTE_LEDGER_DIR = OUT_ROOT / "note_ledger"
RESOLVED_DIR = OUT_ROOT / "resolved_notes"
UNRESOLVED_DIR = OUT_ROOT / "unresolved_notes"
MENTION_DIR = OUT_ROOT / "mentions"
FINAL_DIR = OUT_ROOT / "final_records"
REVIEW_DIR = OUT_ROOT / "review_exports"
RAW_LLM_DIR = OUT_ROOT / "raw_llm_outputs"
TMP_IMG_DIR = OUT_ROOT / "tmp_images"
LOG_DIR = OUT_ROOT / "logs"
OFFLOAD_DIR = OUT_ROOT / "hf_offload"
HF_CACHE_DIR = OUT_ROOT / "hf_cache"
RUN_MANIFEST_PATH = OUT_ROOT / "run_manifest.json"

for p in [
    PAGE_LEDGER_DIR, NOTE_LEDGER_DIR, RESOLVED_DIR, UNRESOLVED_DIR, MENTION_DIR,
    FINAL_DIR, REVIEW_DIR, RAW_LLM_DIR, TMP_IMG_DIR, LOG_DIR, OFFLOAD_DIR, HF_CACHE_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(INPUT_PDF_DIR.glob("*.pdf")) if INPUT_PDF_DIR.exists() else []
RUN_PDF_FILES = pdf_files[:LIMIT_PDFS] if LIMIT_PDFS is not None else pdf_files
RUN_PATIENT_CODES = [p.stem for p in RUN_PDF_FILES]

MODEL_ID = os.environ.get("EVOLET_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")
USE_4BIT = True
LOCAL_FILES_ONLY = False

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
)

SKIP_EXISTING = True

USE_DOCTR_OCR = True
OCR_RENDER_DPI = 130
OCR_BATCH_PAGES = 10
NATIVE_TEXT_MIN_CHARS = 120
NATIVE_TEXT_MIN_WORDS = 22
NATIVE_TEXT_MIN_ALPHA_RATIO = 0.28

NOTE_MIN_WORDS = 12
DEDUP_EXACT_NOTES = True

MAX_NOTES_PER_PATIENT_FOR_LLM = 24
MIN_SIGNAL_FOR_LLM = 3
MIN_REGEX_MENTIONS_TO_SKIP_LLM = 3
SEND_SHORT_NOTES_TO_LLM = False

# Adaptive batching for the 147-file run.
SHORT_NOTE_WORDS = 120
MEDIUM_NOTE_WORDS = 220
LONG_NOTE_WORDS = 380

SHORT_BATCH_SIZE = 3 if torch.cuda.is_available() else 1
MEDIUM_BATCH_SIZE = 2 if torch.cuda.is_available() else 1
LONG_BATCH_SIZE = 1
XLONG_BATCH_SIZE = 1

SMOKE_TEST_BATCH_SIZE = 1
SMOKE_TEST_PATIENTS = 1
SMOKE_TEST_NOTES_PER_PATIENT = 4

# Qwen 2.5 supports much longer generation, but for structured JSON extraction
# 640 is a better default ceiling here than the old 320.
MAX_INPUT_TOKENS = 3584
MAX_NEW_TOKENS = 640
RETRY_MAX_NEW_TOKENS = 768
DO_SAMPLE = False

# Save raw model text by default for pilot, but keep full147 lighter unless forced on.
SAVE_RAW_LLM_OUTPUTS = (
    os.environ.get("EVOLET_SAVE_RAW", "0" if RUN_MODE == "full147" else "1").strip() == "1"
)

print("INPUT_PDF_DIR   :", INPUT_PDF_DIR)
print("PDF count       :", len(pdf_files))
print("RUN_MODE        :", RUN_MODE)
print("RUN_PDF_COUNT   :", len(RUN_PDF_FILES))
print("OUT_ROOT        :", OUT_ROOT)
print("MODEL_ID        :", MODEL_ID)
print("HF token found  :", bool(HF_TOKEN))
print("GPU available   :", torch.cuda.is_available())
print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS)
print("MAX_NEW_TOKENS  :", MAX_NEW_TOKENS)
print("SAVE_RAW_LLM    :", SAVE_RAW_LLM_OUTPUTS)

!df -h


INPUT_PDF_DIR   : /home/pardeep/data/TMH_Patient_Reports
PDF count       : 147
RUN_MODE        : full147
RUN_PDF_COUNT   : 147
OUT_ROOT        : /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147
MODEL_ID        : Qwen/Qwen2.5-1.5B-Instruct
HF token found  : False
GPU available   : True
MAX_INPUT_TOKENS: 3584
MAX_NEW_TOKENS  : 640
SAVE_RAW_LLM    : False
Filesystem       Size  Used Avail Use% Mounted on
/dev/root        290G  290G  5.9M 100% /
tmpfs             16G  4.0K   16G   1% /dev/shm
tmpfs            6.3G  9.1M  6.3G   1% /run
efivarfs         256K   32K  220K  13% /sys/firmware/efi/efivars
tmpfs            5.0M     0  5.0M   0% /run/lock
tmpfs             16G     0   16G   0% /tmp
tmpfs            1.0M     0  1.0M   0% /run/credentials/systemd-journald.service
tmpfs            1.0M     0  1.0M   0% /run/credentials/systemd-resolved.service
/dev/nvme0n1p13  989M   76M  846M   9% /boot
/dev/nvme0n1p15  105M  6.3M   99M   6% /boot/efi
tmpfs            1.0M     0  1.0M   0% /run/

In [ ]:
# Cell 3 — imports, helpers, logger, JSON parsing

import gc
import re
import json
import ast
import time
import math
import hashlib
import shutil
import logging
import traceback
from collections import defaultdict
from typing import Any, Dict, List, Optional

import fitz
import numpy as np
import pandas as pd
import orjson
import torch

from tqdm.auto import tqdm

try:
    from json_repair import repair_json
except Exception:
    repair_json = None

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

def setup_logger(log_path: Path):
    logger = logging.getLogger("evolet_qwen15_optimized")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    return logger

logger = setup_logger(LOG_DIR / "pipeline.log")

def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    payload = orjson.dumps(obj, option=orjson.OPT_INDENT_2)
    with open(tmp_path, "wb") as f:
        f.write(payload)
        f.flush()
        try:
            os.fsync(f.fileno())
        except Exception:
            pass
    os.replace(tmp_path, path)

def load_json(path: Path) -> Any:
    with open(path, "rb") as f:
        raw = f.read()
    return orjson.loads(raw)

def try_load_json(path: Path) -> Optional[Any]:
    try:
        if not path.exists() or path.stat().st_size == 0:
            return None
        with open(path, "rb") as f:
            raw = f.read()
        if not raw or not raw.strip():
            return None
        return orjson.loads(raw)
    except Exception:
        return None

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = str(text).replace("\x00", " ").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def alpha_ratio(text: str) -> float:
    if not text:
        return 0.0
    alpha = sum(ch.isalpha() for ch in text)
    return alpha / max(len(text), 1)

def text_hash(text: str) -> str:
    return hashlib.md5(normalize_text(text).encode("utf-8")).hexdigest()

def release_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def gpu_report():
    if torch.cuda.is_available():
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"CUDA free : {free_b / 1024**3:.2f} GB")
        print(f"CUDA total: {total_b / 1024**3:.2f} GB")

def write_run_manifest():
    manifest = {
        "run_mode": RUN_MODE,
        "limit_pdfs": LIMIT_PDFS,
        "run_pdf_count": len(RUN_PDF_FILES),
        "patient_codes": RUN_PATIENT_CODES,
        "source_pdfs": [p.name for p in RUN_PDF_FILES],
        "out_root": str(OUT_ROOT),
    }
    save_json(manifest, RUN_MANIFEST_PATH)
    return manifest

def load_run_manifest() -> Dict[str, Any]:
    if RUN_MANIFEST_PATH.exists():
        return load_json(RUN_MANIFEST_PATH)
    return write_run_manifest()

def selected_patient_codes() -> List[str]:
    manifest = load_run_manifest()
    return list(manifest.get("patient_codes", []))

def _strip_response_wrappers(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r"^<json>\s*", "", text, flags=re.I)
    text = re.sub(r"\s*</json>$", "", text, flags=re.I)
    text = re.sub(r"^\s*assistant\s*[:\-]\s*", "", text, flags=re.I)
    return text.strip()

def _extract_balanced_json_candidates(text: str) -> List[str]:
    candidates = []
    for open_ch, close_ch in [("{", "}"), ("[", "]")]:
        start_positions = [m.start() for m in re.finditer(re.escape(open_ch), text)]
        for start in start_positions:
            depth = 0
            in_string = False
            escape = False
            for idx in range(start, len(text)):
                ch = text[idx]
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"':
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if ch == open_ch:
                    depth += 1
                elif ch == close_ch:
                    depth -= 1
                    if depth == 0:
                        candidates.append(text[start:idx + 1])
                        break
    seen, uniq = set(), []
    for c in candidates:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            uniq.append(c)
    return uniq

def _normalize_jsonish_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text

def parse_json_loose(text: str) -> Any:
    cleaned = _strip_response_wrappers(text)
    candidates = [cleaned] + _extract_balanced_json_candidates(cleaned)

    for cand in candidates:
        cand = _normalize_jsonish_text(cand)
        for loader in (json.loads, orjson.loads):
            try:
                return loader(cand)
            except Exception:
                pass

    if repair_json is not None:
        for cand in candidates:
            try:
                repaired = repair_json(cand, return_objects=True)
                if isinstance(repaired, (dict, list)):
                    return repaired
            except Exception:
                pass

    for cand in candidates:
        try:
            pyish = cand
            pyish = re.sub(r"\btrue\b", "True", pyish, flags=re.I)
            pyish = re.sub(r"\bfalse\b", "False", pyish, flags=re.I)
            pyish = re.sub(r"\bnull\b", "None", pyish, flags=re.I)
            parsed = ast.literal_eval(pyish)
            if isinstance(parsed, (dict, list)):
                return parsed
        except Exception:
            pass

    preview = cleaned[:500].replace("\n", "\\n")
    raise ValueError(f"Could not parse JSON. Preview: {preview}")

write_run_manifest()
gpu_report()


CUDA free : 19.48 GB
CUDA total: 22.03 GB


In [ ]:
empty_cached_mentions = []

for fp in sorted(MENTION_DIR.glob("*_mentions.json")):
    ok = try_load_json(fp)
    if ok is None:
        empty_cached_mentions.append(fp)

print("Corrupt/empty cached mention files:", len(empty_cached_mentions))
for fp in empty_cached_mentions[:20]:
    print(" -", fp.name)

for fp in empty_cached_mentions:
    try:
        fp.unlink(missing_ok=True)
    except Exception:
        pass

print("Deleted corrupt/empty cached mention files:", len(empty_cached_mentions))

Corrupt/empty cached mention files: 46
 - Hansadevi Basnal_mentions.json
 - Harish_mentions.json
 - Indra Kumar Shukla_mentions.json
 - Irfan Ahmad_mentions.json
 - JALLAPPA JALLAPPA THOKA_mentions.json
 - JANARDAN KUMAR SINGH_mentions.json
 - JAYASHREE DAMODAR JAMBHALE_mentions.json
 - JUMATUN KHATUN_mentions.json
 - KALIKAPRASAD RAMCHARAN VISHWAKARMA_mentions.json
 - KALYAN ROY_mentions.json
 - KAZI ABDUL SATTAR_mentions.json
 - KOMAL KACHHAP_mentions.json
 - KRISHNA PADA SARDAR_mentions.json
 - LALTU SHEIKH_mentions.json
 - Laxmi Wagh_mentions.json
 - MADHUSOODHANAN BHASKARAN PILLAI_mentions.json
 - MAHESH VIJAYVARGIYA_mentions.json
 - MANOJ KUMAR_mentions.json
 - MAYUR NAVNATHJI KAMBLE_mentions.json
 - MD ABUL KALAM AZAD_mentions.json
Deleted corrupt/empty cached mention files: 46


In [ ]:

# Cell 4 — native extraction, OCR fallback, note segmentation, note triage

from doctr.io import DocumentFile
from doctr.models import ocr_predictor

SECTION_SPLIT_PATTERNS = [
    r"\bDATE\s*:\s*\d{1,2}[./-]\d{1,2}[./-]\d{2,4}",
    r"\bCLINICAL NOTE DETAILS\b",
    r"\bPATIENT ASSESSMENT DETAILS\b",
    r"\bJOINT-CLINIC DETAILS\b",
    r"\bREMARKS DETAILS\b",
    r"\bDETAILS OF JOINT CLINIC\b",
]

LOW_VALUE_PATTERNS = [
    r"prescription has been issued by",
    r"cost certificate issued",
    r"https?://",
    r"tata memorial hospital[- ]electronic medical record",
    r"fax:",
    r"e-mail:",
]

CLINICAL_HINT_PATTERNS = [
    r"\bdiagnosis\b", r"\bcarcinoma\b", r"\brcc\b", r"\besophagus\b",
    r"\bmetast", r"\bstage\b", r"\bgrade\b", r"\bbiopsy\b",
    r"\bhistopath\b", r"\bhpr\b", r"\bihc\b", r"\bpet ct\b",
    r"\bc?ect\b", r"\bmri\b", r"\bradiotherapy\b", r"\brt\b",
    r"\bsurgery\b", r"\bnephrectomy\b", r"\bsunitinib\b",
    r"\bchemo", r"\btablet\b", r"\btab\b", r"\bplan\b",
    r"\bfollow[- ]?up\b", r"\bpain\b", r"\bcreat\b", r"\bhb\b",
    r"\bplatelet\b", r"\blymph", r"\bnodule\b", r"\blesion\b",
]

DATE_PATTERNS = [
    r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
    r"\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b",
]

ocr_engine = None

def init_ocr():
    global ocr_engine
    if not USE_DOCTR_OCR or ocr_engine is not None:
        return
    ocr_engine = ocr_predictor(pretrained=True, assume_straight_pages=True)
    if torch.cuda.is_available():
        ocr_engine = ocr_engine.cuda().half()
    ocr_engine.eval()

def doctr_export_page_to_text(page_export: Dict[str, Any]) -> str:
    lines_out = []
    for block in page_export.get("blocks", []) or []:
        for line in block.get("lines", []) or []:
            words = []
            for word in line.get("words", []) or []:
                val = normalize_text(word.get("value") or word.get("text") or "")
                if val:
                    words.append(val)
            line_text = " ".join(words).strip()
            if line_text:
                lines_out.append(line_text)
    return "\n".join(lines_out).strip()

def ocr_image_paths(image_paths: List[Path]) -> List[str]:
    if not image_paths:
        return []
    init_ocr()
    doc = DocumentFile.from_images([str(p) for p in image_paths])
    result = ocr_engine(doc).export()
    pages = result.get("pages", []) or []
    return [doctr_export_page_to_text(p) for p in pages]

def get_native_page_text(page) -> str:
    parts = []
    try:
        for block in page.get_text("blocks", sort=True):
            if len(block) >= 5 and isinstance(block[4], str):
                t = normalize_text(block[4])
                if t:
                    parts.append(t)
    except Exception:
        pass
    if not parts:
        try:
            t = normalize_text(page.get_text("text", sort=True))
            if t:
                parts.append(t)
        except Exception:
            pass
    return "\n".join(parts).strip()

def page_needs_ocr(text: str) -> bool:
    text = normalize_text(text)
    if len(text) < NATIVE_TEXT_MIN_CHARS:
        return True
    if word_count(text) < NATIVE_TEXT_MIN_WORDS:
        return True
    if alpha_ratio(text) < NATIVE_TEXT_MIN_ALPHA_RATIO:
        return True
    return False

def render_page_to_png(page, out_path: Path, dpi: int = OCR_RENDER_DPI) -> Path:
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    pix.save(str(out_path))
    return out_path

def extract_dates(text: str) -> List[str]:
    hits = []
    for pat in DATE_PATTERNS:
        hits.extend(re.findall(pat, text, flags=re.I))
    seen, out = set(), []
    for x in hits:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def is_low_value_note(text: str) -> bool:
    t = normalize_text(text).lower()
    if word_count(t) < NOTE_MIN_WORDS:
        return True
    hits = sum(1 for pat in LOW_VALUE_PATTERNS if re.search(pat, t, flags=re.I))
    if hits >= 2:
        return True
    if "prescription has been issued" in t and word_count(t) < 45:
        return True
    return False

def clinical_signal_score(text: str) -> int:
    t = normalize_text(text).lower()
    score = 0
    score += min(len(t) // 400, 4)
    score += min(len(extract_dates(t)), 3)
    for pat in CLINICAL_HINT_PATTERNS:
        if re.search(pat, t, flags=re.I):
            score += 1
    return score

def split_into_notes(page_num: int, text: str) -> List[Dict[str, Any]]:
    text = normalize_text(text)
    if not text:
        return []
    split_re = "(" + "|".join(SECTION_SPLIT_PATTERNS) + ")"
    parts = re.split(split_re, text, flags=re.I)
    merged = []
    cur = ""
    for chunk in parts:
        chunk = normalize_text(chunk)
        if not chunk:
            continue
        if re.match(split_re, chunk, flags=re.I):
            if cur.strip():
                merged.append(cur.strip())
            cur = chunk
        else:
            cur = (cur + "\n" + chunk).strip() if cur else chunk
    if cur.strip():
        merged.append(cur.strip())

    out = []
    for i, note_text in enumerate(merged, start=1):
        note_text = normalize_text(note_text)
        if word_count(note_text) < NOTE_MIN_WORDS:
            continue
        out.append({
            "page_num": page_num,
            "note_ix": i,
            "note_id": f"p{page_num:04d}_n{i:03d}",
            "text": note_text,
            "hash": text_hash(note_text),
            "dates": extract_dates(note_text),
            "signal": clinical_signal_score(note_text),
            "low_value": is_low_value_note(note_text),
        })
    return out

print("Loaded extraction and note segmentation helpers.")


Loaded extraction and note segmentation helpers.


## *This block is where the pipeline becomes lossless-first. Instead of sending raw PDFs straight into an LLM, it first builds stable intermediate layers: page-level text and note-level text. That is the right design because it makes the workflow resumable, auditable, and much cheaper than full-document prompting. Native extraction is attempted before OCR because OCR is slower, noisier, and more expensive in compute; it is only used when the page appears weak. Better OCR alternatives could be PaddleOCR, Tesseract, EasyOCR, Surya, or cloud OCR services, and better segmentation could come from layout-aware parsers or document transformers. Still, the chosen mix makes sense here because it balances speed, local VM execution, and acceptable accuracy for a large 147-PDF batch. The cleanup cell also reflects a real operational fix: interrupted runs had produced empty cache files, so the notebook now cleans those before reuse.*

In [ ]:

# Cell 5 — pass1: build page ledgers and note ledgers

def process_pdf_to_ledgers(pdf_path: Path) -> Dict[str, Any]:
    patient_code = pdf_path.stem
    page_out_path = PAGE_LEDGER_DIR / f"{patient_code}_pages.json"
    note_out_path = NOTE_LEDGER_DIR / f"{patient_code}_notes.json"

    if SKIP_EXISTING and page_out_path.exists() and note_out_path.exists():
        return {
            "page_doc": load_json(page_out_path),
            "note_doc": load_json(note_out_path),
        }

    doc = fitz.open(pdf_path)
    pdf_tmp_dir = TMP_IMG_DIR / patient_code
    pdf_tmp_dir.mkdir(parents=True, exist_ok=True)

    page_rows = []
    weak_page_indices = []

    for i in range(len(doc)):
        page = doc[i]
        page_num = i + 1
        native_text = normalize_text(get_native_page_text(page))
        row = {
            "page_num": page_num,
            "selected_source": "native",
            "text": native_text,
            "char_count": len(native_text),
            "word_count": word_count(native_text),
            "alpha_ratio": round(alpha_ratio(native_text), 4),
            "need_ocr": False,
        }
        if USE_DOCTR_OCR and page_needs_ocr(native_text):
            img_path = pdf_tmp_dir / f"page_{page_num:04d}.png"
            render_page_to_png(page, img_path, dpi=OCR_RENDER_DPI)
            row["need_ocr"] = True
            row["tmp_image_path"] = str(img_path)
            weak_page_indices.append(len(page_rows))
        page_rows.append(row)

    if USE_DOCTR_OCR and weak_page_indices:
        weak_paths = [Path(page_rows[idx]["tmp_image_path"]) for idx in weak_page_indices]
        for start in range(0, len(weak_paths), OCR_BATCH_PAGES):
            batch_paths = weak_paths[start:start + OCR_BATCH_PAGES]
            batch_texts = ocr_image_paths(batch_paths)
            batch_idxs = weak_page_indices[start:start + OCR_BATCH_PAGES]
            for j, idx in enumerate(batch_idxs):
                ocr_text = normalize_text(batch_texts[j] if j < len(batch_texts) else "")
                if len(ocr_text) > len(page_rows[idx]["text"]):
                    page_rows[idx]["text"] = ocr_text
                    page_rows[idx]["char_count"] = len(ocr_text)
                    page_rows[idx]["word_count"] = word_count(ocr_text)
                    page_rows[idx]["alpha_ratio"] = round(alpha_ratio(ocr_text), 4)
                    page_rows[idx]["selected_source"] = "ocr"

    for row in page_rows:
        row.pop("tmp_image_path", None)

    notes = []
    seen_hashes = set()
    for row in page_rows:
        for note in split_into_notes(row["page_num"], row["text"]):
            if DEDUP_EXACT_NOTES and note["hash"] in seen_hashes:
                continue
            seen_hashes.add(note["hash"])
            notes.append(note)

    page_doc = {
        "patient_code": patient_code,
        "source_pdf": pdf_path.name,
        "page_count": len(page_rows),
        "pages": page_rows,
    }
    note_doc = {
        "patient_code": patient_code,
        "source_pdf": pdf_path.name,
        "note_count": len(notes),
        "notes": notes,
    }

    save_json(page_doc, page_out_path)
    save_json(note_doc, note_out_path)
    shutil.rmtree(pdf_tmp_dir, ignore_errors=True)

    return {"page_doc": page_doc, "note_doc": note_doc}

pass1_rows, pass1_failures = [], []
run_pdf_files = pdf_files[:LIMIT_PDFS] if LIMIT_PDFS is not None else pdf_files
print("PDF files to process:", len(run_pdf_files))

for pdf_path in tqdm(run_pdf_files, desc="Pass1 ledgers"):
    try:
        res = process_pdf_to_ledgers(pdf_path)
        page_doc, note_doc = res["page_doc"], res["note_doc"]
        pages = page_doc["pages"]
        notes = note_doc["notes"]
        pass1_rows.append({
            "patient_code": page_doc["patient_code"],
            "source_pdf": page_doc["source_pdf"],
            "page_count": page_doc["page_count"],
            "ocr_pages": sum(1 for p in pages if p["selected_source"] == "ocr"),
            "note_count": note_doc["note_count"],
            "low_value_notes": sum(1 for n in notes if n["low_value"]),
            "high_signal_notes": sum(1 for n in notes if n["signal"] >= MIN_SIGNAL_FOR_LLM),
            "status": "ok",
        })
    except Exception as e:
        pass1_failures.append({"file": pdf_path.name, "error": repr(e), "traceback": traceback.format_exc()})
        logger.exception(f"Pass1 failed: {pdf_path.name}")
    finally:
        release_cuda()

pass1_df = pd.DataFrame(pass1_rows)
pass1_fail_df = pd.DataFrame(pass1_failures)
pass1_df.to_csv(REVIEW_DIR / "pass1_inventory.csv", index=False)
pass1_fail_df.to_csv(REVIEW_DIR / "pass1_failures.csv", index=False)
display(pass1_df.head(20))
display(pass1_fail_df.head(10))


PDF files to process: 147


Pass1 ledgers:   0%|          | 0/147 [00:00<?, ?it/s]

,patient_code,source_pdf,page_count,ocr_pages,note_count,low_value_notes,high_signal_notes,status
0,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,10,0,59,10,25,ok
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,49,0,121,43,81,ok
2,ABDUL MABUD,ABDUL MABUD.pdf,32,0,80,33,49,ok
3,ABDUL RAJAK KAJI,ABDUL RAJAK KAJI.pdf,13,0,70,23,21,ok
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,75,0,175,11,52,ok
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,284,0,598,47,172,ok
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,43,0,239,86,86,ok
7,AMIR AL MORSHED,AMIR AL MORSHED.pdf,16,0,48,21,31,ok
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,102,0,331,28,79,ok
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,45,0,266,53,82,ok


""


In [ ]:

# Cell 6 — deterministic extractor for easy notes

LINE_BREAK_RE = re.compile(r"\n+")

def first_date_or_blank(text: str) -> str:
    ds = extract_dates(text)
    return ds[0] if ds else ""

def add_mention(out: List[Dict[str, Any]], note: Dict[str, Any], category: str,
                label: str, value: str, certainty: str = "confirmed",
                attributes: Optional[Dict[str, Any]] = None, evidence_quote: Optional[str] = None):
    value = normalize_text(value)
    label = normalize_text(label)
    if not value:
        return
    out.append({
        "category": category or "unknown",
        "label": label,
        "value": value,
        "normalized_value": value,
        "date_text": first_date_or_blank(note["text"]),
        "certainty": certainty,
        "attributes": attributes or {},
        "source_pages": [note["page_num"]],
        "evidence_ids": [note["note_id"]],
        "evidence_quote": normalize_text(evidence_quote or value[:240]),
        "origin": "regex",
    })

def extract_plan_block(text: str) -> str:
    lines = [normalize_text(x) for x in LINE_BREAK_RE.split(text)]
    lines = [x for x in lines if x]
    out = []
    capture = False
    for line in lines:
        low = line.lower()
        if re.match(r"^(plan|tentative plan|adv|remarks / others)\s*[:\-]?$", low):
            capture = True
            continue
        if capture:
            if re.match(r"^(date|clinical note details|patient assessment details|joint-clinic details|entered by|seen by)\b", low):
                break
            out.append(line)
            if len(out) >= 6:
                break
    return normalize_text(" | ".join(out))

def simple_extract_note_mentions(note: Dict[str, Any]) -> List[Dict[str, Any]]:
    text = note["text"]
    low = text.lower()
    mentions = []

    for pat in [
        r"final diagnosis\s*[:\-]\s*(.+)",
        r"diagnosis\s*[:\-]\s*(.+)",
        r"imp\s*[-:]\s*(.+)",
        r"impression\s*[:\-]\s*(.+)",
        r"case of\s+(.+)",
    ]:
        m = re.search(pat, text, flags=re.I)
        if m:
            val = normalize_text(m.group(1))
            val = re.split(r"\n|\|", val)[0]
            add_mention(mentions, note, "diagnosis", "diagnosis", val, evidence_quote=val)
            break

    for pat, label in [
        (r"(hpr[^\n:]*[:\-]\s*.+)", "HPR"),
        (r"(histopath[^\n:]*[:\-]\s*.+)", "histopathology"),
        (r"(ihc[^\n:]*[:\-]\s*.+)", "IHC"),
    ]:
        for m in re.finditer(pat, text, flags=re.I):
            val = normalize_text(m.group(1))
            add_mention(mentions, note, "pathology", label, val, evidence_quote=val)

    for pat, label in [
        (r"(pet ct[^\n]*[:\-]?\s*.+)", "PET CT"),
        (r"(c?ect[^\n]*[:\-]?\s*.+)", "CECT"),
        (r"(mri[^\n]*[:\-]?\s*.+)", "MRI"),
        (r"(ct study reveals[^\n]*[:\-]?\s*.+)", "CT"),
    ]:
        for m in re.finditer(pat, text, flags=re.I):
            val = normalize_text(m.group(1))
            add_mention(mentions, note, "imaging", label, val, evidence_quote=val)

    for m in re.finditer(r"((tab|t\.)\s*[A-Z][A-Za-z0-9.+-]*(?:\s+[A-Za-z0-9.+-]+){0,4}\s*(?:\d+(?:\.\d+)?)?\s*(?:mg|mcg|gm)?[^\n]*)", text, flags=re.I):
        val = normalize_text(m.group(1))
        if len(val) >= 5:
            add_mention(mentions, note, "medication", "medication", val, evidence_quote=val)

    for drug in ["sunitinib", "everolimus", "pazopanib", "nivolumab", "cabozantinib", "chemotherapy"]:
        if re.search(rf"\b{drug}\b", low):
            line = next((ln.strip() for ln in text.splitlines() if drug in ln.lower()), drug)
            add_mention(mentions, note, "medication", "drug", line, evidence_quote=line)

    for m in re.finditer(r"(c/o\s*[-:]?\s*[^\n]+)", text, flags=re.I):
        add_mention(mentions, note, "symptom", "complaint", m.group(1), evidence_quote=m.group(1))
    for m in re.finditer(r"(pain[^\n]{0,120})", text, flags=re.I):
        add_mention(mentions, note, "symptom", "pain", m.group(1), evidence_quote=m.group(1))

    for m in re.finditer(r"\b(PS\s*[-:]?\s*\d)\b", text, flags=re.I):
        add_mention(mentions, note, "performance_status", "PS", m.group(1), evidence_quote=m.group(1))
    for m in re.finditer(r"\b(KPS\s*\d{2,3})\b", text, flags=re.I):
        add_mention(mentions, note, "performance_status", "KPS", m.group(1), evidence_quote=m.group(1))

    for key, category in [("nephrectomy", "surgery"), ("surgery", "surgery"), ("rt", "radiotherapy"), ("radiotherapy", "radiotherapy"), ("sbrt", "radiotherapy")]:
        if re.search(rf"\b{re.escape(key)}\b", low):
            line = next((ln.strip() for ln in text.splitlines() if key in ln.lower()), key)
            add_mention(mentions, note, category, key, line, evidence_quote=line)

    for m in re.finditer(r"\b(serum creat\s*\d+(?:\.\d+)?)\b", low, flags=re.I):
        add_mention(mentions, note, "lab", "serum creatinine", m.group(1), evidence_quote=m.group(1))
    for m in re.finditer(r"\b(NGS\s*[:\-]?\s*[^\n]+)\b", text, flags=re.I):
        add_mention(mentions, note, "genomics", "NGS", m.group(1), evidence_quote=m.group(1))

    plan_block = extract_plan_block(text)
    if plan_block:
        add_mention(mentions, note, "plan", "plan", plan_block, evidence_quote=plan_block)

    for m in re.finditer(r"(review after[^\n]+|r/s[^\n]+|follow[- ]?up[^\n]+)", text, flags=re.I):
        add_mention(mentions, note, "follow_up", "follow_up", m.group(1), evidence_quote=m.group(1))

    seen, out = set(), []
    for x in mentions:
        key = (x["category"], x["label"].lower(), x["value"].lower())
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out

def note_needs_llm(note: Dict[str, Any], regex_mentions: List[Dict[str, Any]]) -> bool:
    if note.get("low_value"):
        return False
    if note.get("signal", 0) < MIN_SIGNAL_FOR_LLM:
        return False
    if regex_mentions and len(regex_mentions) >= MIN_REGEX_MENTIONS_TO_SKIP_LLM:
        return False
    if not SEND_SHORT_NOTES_TO_LLM and word_count(note["text"]) < 40:
        return False
    return True

print("Loaded deterministic extraction and triage.")


Loaded deterministic extraction and triage.


In [ ]:
# Cell 7 — run triage and cache easy-note outputs

# Auto-heal after kernel restart: if Cell 6 was not run in this kernel,
# rebuild the deterministic triage helpers locally before triage starts.
if "simple_extract_note_mentions" not in globals() or "note_needs_llm" not in globals():
    print("Rebuilding deterministic triage helpers inside Cell 7 because Cell 6 is missing in this kernel.")

    LINE_BREAK_RE = re.compile(r"\n+")

    def first_date_or_blank(text: str) -> str:
        ds = extract_dates(text)
        return ds[0] if ds else ""

    def add_mention(out: List[Dict[str, Any]], note: Dict[str, Any], category: str,
                    label: str, value: str, certainty: str = "confirmed",
                    attributes: Optional[Dict[str, Any]] = None, evidence_quote: Optional[str] = None):
        value = normalize_text(value)
        label = normalize_text(label)
        if not value:
            return
        out.append({
            "category": category or "unknown",
            "label": label,
            "value": value,
            "normalized_value": value,
            "date_text": first_date_or_blank(note["text"]),
            "certainty": certainty,
            "attributes": attributes or {},
            "source_pages": [note["page_num"]],
            "evidence_ids": [note["note_id"]],
            "evidence_quote": normalize_text(evidence_quote or value[:240]),
            "origin": "regex",
        })

    def extract_plan_block(text: str) -> str:
        lines = [normalize_text(x) for x in LINE_BREAK_RE.split(text)]
        lines = [x for x in lines if x]
        out = []
        capture = False
        for line in lines:
            low = line.lower()
            if re.match(r"^(plan|tentative plan|adv|remarks / others)\s*[:\-]?$", low):
                capture = True
                continue
            if capture:
                if re.match(r"^(date|clinical note details|patient assessment details|joint-clinic details|entered by|seen by)\b", low):
                    break
                out.append(line)
                if len(out) >= 6:
                    break
        return normalize_text(" | ".join(out))

    def simple_extract_note_mentions(note: Dict[str, Any]) -> List[Dict[str, Any]]:
        text = note["text"]
        low = text.lower()
        mentions = []

        for pat in [
            r"final diagnosis\s*[:\-]\s*(.+)",
            r"diagnosis\s*[:\-]\s*(.+)",
            r"imp\s*[-:]\s*(.+)",
            r"impression\s*[:\-]\s*(.+)",
            r"case of\s+(.+)",
        ]:
            m = re.search(pat, text, flags=re.I)
            if m:
                val = normalize_text(m.group(1))
                val = re.split(r"\n|\|", val)[0]
                add_mention(mentions, note, "diagnosis", "diagnosis", val, evidence_quote=val)
                break

        for pat, label in [
            (r"(hpr[^\n:]*[:\-]\s*.+)", "HPR"),
            (r"(histopath[^\n:]*[:\-]\s*.+)", "histopathology"),
            (r"(ihc[^\n:]*[:\-]\s*.+)", "IHC"),
        ]:
            for m in re.finditer(pat, text, flags=re.I):
                val = normalize_text(m.group(1))
                add_mention(mentions, note, "pathology", label, val, evidence_quote=val)

        for pat, label in [
            (r"(pet ct[^\n]*[:\-]?\s*.+)", "PET CT"),
            (r"(c?ect[^\n]*[:\-]?\s*.+)", "CECT"),
            (r"(mri[^\n]*[:\-]?\s*.+)", "MRI"),
            (r"(ct study reveals[^\n]*[:\-]?\s*.+)", "CT"),
        ]:
            for m in re.finditer(pat, text, flags=re.I):
                val = normalize_text(m.group(1))
                add_mention(mentions, note, "imaging", label, val, evidence_quote=val)

        for m in re.finditer(r"((tab|t\.)\s*[A-Z][A-Za-z0-9.+-]*(?:\s+[A-Za-z0-9.+-]+){0,4}\s*(?:\d+(?:\.\d+)?)?\s*(?:mg|mcg|gm)?[^\n]*)", text, flags=re.I):
            val = normalize_text(m.group(1))
            if len(val) >= 5:
                add_mention(mentions, note, "medication", "medication", val, evidence_quote=val)

        for drug in ["sunitinib", "everolimus", "pazopanib", "nivolumab", "cabozantinib", "chemotherapy"]:
            if re.search(rf"\b{drug}\b", low):
                line = next((ln.strip() for ln in text.splitlines() if drug in ln.lower()), drug)
                add_mention(mentions, note, "medication", "drug", line, evidence_quote=line)

        for m in re.finditer(r"(c/o\s*[-:]?\s*[^\n]+)", text, flags=re.I):
            add_mention(mentions, note, "symptom", "complaint", m.group(1), evidence_quote=m.group(1))
        for m in re.finditer(r"(pain[^\n]{0,120})", text, flags=re.I):
            add_mention(mentions, note, "symptom", "pain", m.group(1), evidence_quote=m.group(1))

        for m in re.finditer(r"\b(PS\s*[-:]?\s*\d)\b", text, flags=re.I):
            add_mention(mentions, note, "performance_status", "PS", m.group(1), evidence_quote=m.group(1))
        for m in re.finditer(r"\b(KPS\s*\d{2,3})\b", text, flags=re.I):
            add_mention(mentions, note, "performance_status", "KPS", m.group(1), evidence_quote=m.group(1))

        for key, category in [
            ("nephrectomy", "surgery"),
            ("surgery", "surgery"),
            ("rt", "radiotherapy"),
            ("radiotherapy", "radiotherapy"),
            ("sbrt", "radiotherapy")
        ]:
            if re.search(rf"\b{re.escape(key)}\b", low):
                line = next((ln.strip() for ln in text.splitlines() if key in ln.lower()), key)
                add_mention(mentions, note, category, key, line, evidence_quote=line)

        for m in re.finditer(r"\b(serum creat\s*\d+(?:\.\d+)?)\b", low, flags=re.I):
            add_mention(mentions, note, "lab", "serum creatinine", m.group(1), evidence_quote=m.group(1))
        for m in re.finditer(r"\b(NGS\s*[:\-]?\s*[^\n]+)\b", text, flags=re.I):
            add_mention(mentions, note, "genomics", "NGS", m.group(1), evidence_quote=m.group(1))

        plan_block = extract_plan_block(text)
        if plan_block:
            add_mention(mentions, note, "plan", "plan", plan_block, evidence_quote=plan_block)

        for m in re.finditer(r"(review after[^\n]+|r/s[^\n]+|follow[- ]?up[^\n]+)", text, flags=re.I):
            add_mention(mentions, note, "follow_up", "follow_up", m.group(1), evidence_quote=m.group(1))

        seen, out = set(), []
        for x in mentions:
            key = (x["category"], x["label"].lower(), x["value"].lower())
            if key not in seen:
                seen.add(key)
                out.append(x)
        return out

    def note_needs_llm(note: Dict[str, Any], regex_mentions: List[Dict[str, Any]]) -> bool:
        if note.get("low_value"):
            return False
        if note.get("signal", 0) < MIN_SIGNAL_FOR_LLM:
            return False
        if regex_mentions and len(regex_mentions) >= MIN_REGEX_MENTIONS_TO_SKIP_LLM:
            return False
        if not SEND_SHORT_NOTES_TO_LLM and word_count(note["text"]) < 40:
            return False
        return True

triage_rows = []
triage_failures = []

note_files = sorted(NOTE_LEDGER_DIR.glob("*_notes.json"))
print("Note ledgers:", len(note_files))

for fp in tqdm(note_files, desc="Triage notes"):
    try:
        note_doc = load_json(fp)
        patient_code = note_doc["patient_code"]
        resolved_path = RESOLVED_DIR / f"{patient_code}_resolved.json"
        unresolved_path = UNRESOLVED_DIR / f"{patient_code}_unresolved.json"

        if SKIP_EXISTING and resolved_path.exists() and unresolved_path.exists():
            resolved_doc = load_json(resolved_path)
            unresolved_doc = load_json(unresolved_path)
        else:
            resolved_notes, unresolved_notes = [], []
            for note in note_doc["notes"]:
                regex_mentions = simple_extract_note_mentions(note)
                if note_needs_llm(note, regex_mentions):
                    unresolved_notes.append({**note, "regex_mentions": regex_mentions})
                else:
                    resolved_notes.append({**note, "mentions": regex_mentions})

            unresolved_notes = sorted(
                unresolved_notes,
                key=lambda x: (x.get("signal", 0), word_count(x["text"])),
                reverse=True
            )[:MAX_NOTES_PER_PATIENT_FOR_LLM]

            resolved_doc = {
                "patient_code": patient_code,
                "source_pdf": note_doc["source_pdf"],
                "resolved_count": len(resolved_notes),
                "resolved_notes": resolved_notes,
            }
            unresolved_doc = {
                "patient_code": patient_code,
                "source_pdf": note_doc["source_pdf"],
                "unresolved_count": len(unresolved_notes),
                "unresolved_notes": unresolved_notes,
            }
            save_json(resolved_doc, resolved_path)
            save_json(unresolved_doc, unresolved_path)

        triage_rows.append({
            "patient_code": patient_code,
            "source_pdf": note_doc["source_pdf"],
            "note_count": note_doc["note_count"],
            "resolved_notes": resolved_doc["resolved_count"],
            "unresolved_notes": unresolved_doc["unresolved_count"],
            "status": "ok",
        })
    except Exception as e:
        triage_failures.append({
            "file": fp.name,
            "error": repr(e),
            "traceback": traceback.format_exc()
        })
        logger.exception(f"Triage failed: {fp.name}")

triage_df = pd.DataFrame(triage_rows)
triage_fail_df = pd.DataFrame(triage_failures)

triage_df.to_csv(REVIEW_DIR / "triage_inventory.csv", index=False)
triage_fail_df.to_csv(REVIEW_DIR / "triage_failures.csv", index=False)

display(triage_df.head(20))
display(triage_fail_df.head(10))


Note ledgers: 147


Triage notes:   0%|          | 0/147 [00:00<?, ?it/s]

,patient_code,source_pdf,note_count,resolved_notes,unresolved_notes,status
0,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,59,57,2,ok
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,121,112,9,ok
2,ABDUL MABUD,ABDUL MABUD.pdf,80,72,8,ok
3,ABDUL RAJAK KAJI,ABDUL RAJAK KAJI.pdf,70,66,4,ok
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,175,164,11,ok
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,598,561,24,ok
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,239,209,24,ok
7,AMIR AL MORSHED,AMIR AL MORSHED.pdf,48,39,9,ok
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,331,314,17,ok
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,266,250,16,ok


""


## This is the most important architecture choice in the notebook: regex first, LLM second. The notebook intentionally solves easy notes with deterministic rules and only sends unresolved, clinically dense notes to the model. That is a strong choice because medical notes often repeat standard phrases where regex is faster, cheaper, and easier to trust than generative extraction. The self-healing logic in Cell 7 is also practical because notebook kernels restart often; rebuilding helper functions inside triage prevents a full rerun after a partial interruption. On the modeling side, the notebook uses a relatively small local Qwen model with 4-bit loading because the goal is not open-ended reasoning but structured mention rescue at scale. Better alternatives could be larger models, vLLM serving, MedGemma, or API-based long-context models, but the selected local Qwen setup fits the notebook’s constraints: lower VRAM, offline/local control, and predictable batch throughput.

In [ ]:
# Cell 8 — load local model

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

def detect_compute_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

COMPUTE_DTYPE = detect_compute_dtype()
print("COMPUTE_DTYPE:", COMPUTE_DTYPE)

def load_llm(model_id: str):
    common_kwargs = {
        "cache_dir": str(HF_CACHE_DIR),
        "local_files_only": LOCAL_FILES_ONLY,
    }
    if HF_TOKEN:
        common_kwargs["token"] = HF_TOKEN

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        **common_kwargs,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    quant_config = None
    if USE_4BIT and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        torch_dtype=COMPUTE_DTYPE,
        device_map="auto",
        offload_folder=str(OFFLOAD_DIR),
        offload_state_dict=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
        **common_kwargs,
    )
    model.eval()
    model.config.use_cache = True

    gen_cfg = GenerationConfig.from_model_config(model.config)
    gen_cfg.do_sample = DO_SAMPLE
    gen_cfg.max_new_tokens = MAX_NEW_TOKENS
    gen_cfg.use_cache = True
    gen_cfg.pad_token_id = tokenizer.pad_token_id
    gen_cfg.eos_token_id = tokenizer.eos_token_id
    model.generation_config = gen_cfg
    return tokenizer, model

tokenizer, model = load_llm(MODEL_ID)
print("Loaded:", MODEL_ID)
gpu_report()


COMPUTE_DTYPE: torch.bfloat16


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
CUDA free : 17.75 GB
CUDA total: 22.03 GB


In [ ]:
# Cell 9 — fast batch generation helpers

SYSTEM_PROMPT = """
You extract structured clinical mentions from one medical note.

Return exactly one JSON object with this shape:
{
  "mentions": [
    {
      "category": "",
      "label": "",
      "value": "",
      "normalized_value": "",
      "date_text": "",
      "certainty": "confirmed|possible|ruled_out|unknown",
      "attributes": {},
      "evidence_quote": ""
    }
  ]
}

Rules:
- JSON only. No markdown. No commentary.
- Max 8 mentions.
- Do not invent facts.
- evidence_quote must be a short verbatim snippet from the note.
- If no clinically useful mention exists, return {"mentions":[]}.
- Allowed categories:
  diagnosis, medication, plan, symptom, imaging, pathology, genomics,
  procedure, lab, status, follow_up, surgery, radiotherapy,
  performance_status, other
""".strip()

def first_model_device():
    return next(model.parameters()).device

def trim_note_text_for_prompt(text: str, max_chars: int = 9000) -> str:
    text = normalize_text(text)
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.65)
    tail = max_chars - head - 32
    return (
        text[:head].rstrip()
        + "\n\n...[middle truncated for token budget]...\n\n"
        + text[-tail:].lstrip()
    )

def note_word_len(note: Dict[str, Any]) -> int:
    return word_count(note.get("text", ""))

def note_length_bucket(note: Dict[str, Any]) -> str:
    wc = note_word_len(note)
    if wc <= SHORT_NOTE_WORDS:
        return "short"
    if wc <= MEDIUM_NOTE_WORDS:
        return "medium"
    if wc <= LONG_NOTE_WORDS:
        return "long"
    return "xlong"

def batch_size_for_bucket(bucket: str) -> int:
    if bucket == "short":
        return SHORT_BATCH_SIZE
    if bucket == "medium":
        return MEDIUM_BATCH_SIZE
    if bucket == "long":
        return LONG_BATCH_SIZE
    return XLONG_BATCH_SIZE

def max_new_tokens_for_bucket(bucket: str) -> int:
    return RETRY_MAX_NEW_TOKENS if bucket in {"long", "xlong"} else MAX_NEW_TOKENS

def build_llm_prompt(note: Dict[str, Any]) -> str:
    note_text = trim_note_text_for_prompt(note["text"])
    return f"""
note_id={note["note_id"]}
page_num={note["page_num"]}
regex_mentions_already_found={len(note.get("regex_mentions", []))}

Extract clinically useful mentions from this note and return JSON only.

note_text:
{note_text}
""".strip()

def make_generation_batches(notes: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not notes:
        return []
    ordered = sorted(
        notes,
        key=lambda n: (
            ["short", "medium", "long", "xlong"].index(note_length_bucket(n)),
            note_word_len(n),
        ),
    )
    out = []
    current = []
    current_bucket = None
    for note in ordered:
        bucket = note_length_bucket(note)
        limit = batch_size_for_bucket(bucket)
        if current and (bucket != current_bucket or len(current) >= limit):
            out.append(
                {
                    "bucket": current_bucket,
                    "batch_size": batch_size_for_bucket(current_bucket),
                    "max_new_tokens": max_new_tokens_for_bucket(current_bucket),
                    "notes": current,
                }
            )
            current = []
        current_bucket = bucket
        current.append(note)
    if current:
        out.append(
            {
                "bucket": current_bucket,
                "batch_size": batch_size_for_bucket(current_bucket),
                "max_new_tokens": max_new_tokens_for_bucket(current_bucket),
                "notes": current,
            }
        )
    return out

def generate_batch_json(
    prompts: List[str],
    max_input_tokens: int = MAX_INPUT_TOKENS,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> List[str]:
    rendered_batch = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": p},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for p in prompts
    ]

    inputs = tokenizer(
        rendered_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_input_tokens,
    )
    padded_input_len = inputs["input_ids"].shape[1]
    device = first_model_device()
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start_t = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    took = time.time() - start_t
    print(
        f"batch={len(prompts)} prompt_tokens={padded_input_len} "
        f"new={max_new_tokens} time={took:.1f}s"
    )

    decoded = []
    for i in range(outputs.shape[0]):
        new_tokens = outputs[i][padded_input_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        decoded.append(text)
    return decoded

def parse_and_clean_mentions(raw_text: str, note: Dict[str, Any]) -> List[Dict[str, Any]]:
    parsed = parse_json_loose(raw_text)
    mentions = parsed.get("mentions", []) if isinstance(parsed, dict) else []
    cleaned = [clean_llm_mention(x, note) for x in mentions]
    return [x for x in cleaned if x is not None]

def clean_llm_mention(raw: Dict[str, Any], note: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(raw, dict):
        return None
    value = normalize_text(raw.get("value", ""))
    label = normalize_text(raw.get("label", ""))
    if not value and not label:
        return None
    category = normalize_text(raw.get("category", "")) or "unknown"
    certainty = str(raw.get("certainty", "unknown")).strip().lower()
    if certainty not in {"confirmed", "possible", "ruled_out", "unknown"}:
        certainty = "unknown"
    return {
        "category": category,
        "label": label or category,
        "value": value or label,
        "normalized_value": normalize_text(raw.get("normalized_value", "")) or value or label,
        "date_text": normalize_text(raw.get("date_text", "")) or first_date_or_blank(note["text"]),
        "certainty": certainty,
        "attributes": raw.get("attributes", {}) if isinstance(raw.get("attributes", {}), dict) else {},
        "source_pages": [note["page_num"]],
        "evidence_ids": [note["note_id"]],
        "evidence_quote": normalize_text(raw.get("evidence_quote", ""))[:240],
        "origin": "llm",
    }


In [ ]:
# Cell 10 — smoke test one unresolved patient before full LLM rescue

pilot_patient_codes = set(selected_patient_codes())
sample_unresolved_files = [
    fp for fp in sorted(UNRESOLVED_DIR.glob("*_unresolved.json"))
    if fp.stem.replace("_unresolved", "") in pilot_patient_codes
][:SMOKE_TEST_PATIENTS]

print("Smoke test files:", [x.name for x in sample_unresolved_files])

smoke_rows = []
smoke_failures = []

for fp in sample_unresolved_files:
    try:
        doc = load_json(fp)
        notes = doc["unresolved_notes"][: min(SMOKE_TEST_NOTES_PER_PATIENT, len(doc["unresolved_notes"]))]
        raw_outputs = []
        mention_count = 0

        for batch_info in make_generation_batches(notes):
            batch_notes = batch_info["notes"]
            prompts = [build_llm_prompt(n) for n in batch_notes]
            outputs = generate_batch_json(
                prompts,
                max_new_tokens=batch_info["max_new_tokens"],
            )

            for note, raw_text in zip(batch_notes, outputs):
                raw_outputs.append(
                    {
                        "note_id": note["note_id"],
                        "page_num": note["page_num"],
                        "raw_text": raw_text,
                    }
                )
                cleaned = parse_and_clean_mentions(raw_text, note)
                mention_count += len(cleaned)

        if SAVE_RAW_LLM_OUTPUTS:
            save_json(
                {
                    "patient_code": doc["patient_code"],
                    "source_pdf": doc["source_pdf"],
                    "smoke_test_notes": [n["note_id"] for n in notes],
                    "raw_outputs": raw_outputs,
                },
                RAW_LLM_DIR / "smoke_test" / f"{doc['patient_code']}_smoke_outputs.json"
            )

        smoke_rows.append({
            "patient_code": doc["patient_code"],
            "unresolved_test_notes": len(notes),
            "llm_mentions": mention_count,
            "status": "ok",
        })
    except Exception as e:
        smoke_failures.append({
            "file": fp.name,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        logger.exception(f"Smoke test failed: {fp.name}")
    finally:
        release_cuda()

display(pd.DataFrame(smoke_rows))
display(pd.DataFrame(smoke_failures))


Smoke test files: ['. RAMSHARAN SHRIVASTAVA_unresolved.json']
batch=2 prompt_tokens=435 new=640 time=8.8s


,patient_code,unresolved_test_notes,llm_mentions,status
0,. RAMSHARAN SHRIVASTAVA,2,2,ok


""


### This block is the notebook’s controlled generation layer. The prompt is tightly constrained to JSON mention extraction, which is appropriate because the notebook does not want summarization drift or narrative outputs. Adaptive batching is used because short notes can be processed in larger batches, while long notes need safer, smaller batches to avoid memory pressure and unstable outputs. The smoke test is a smart operational safeguard: it catches formatting problems before spending time on the full unresolved set. Better alternatives here would be schema-constrained decoding, JSON mode from newer runtimes, guardrail frameworks, or a serving engine like vLLM for higher throughput. But the current choice is understandable because it keeps everything inside one notebook, works with the local HF stack, and still includes rescue logic like parse_json_loose, retries, and safer cache reuse.

In [ ]:
# Cell 11 — full LLM rescue pass on unresolved notes for the selected run only

def run_llm_on_unresolved_file(fp: Path) -> Dict[str, Any]:
    doc = load_json(fp)
    patient_code = doc["patient_code"]
    out_path = MENTION_DIR / f"{patient_code}_mentions.json"

    if SKIP_EXISTING and out_path.exists():
        cached = try_load_json(out_path)
        if isinstance(cached, dict) and "mentions" in cached:
            return cached
        logger.warning(f"Ignoring corrupt/empty cached mentions file and recomputing: {out_path.name}")
        try:
            out_path.unlink(missing_ok=True)
        except Exception:
            pass

    unresolved = sorted(doc["unresolved_notes"], key=lambda x: (note_length_bucket(x), note_word_len(x)))
    llm_mentions = []
    failures = []
    raw_outputs = []

    for batch_info in make_generation_batches(unresolved):
        batch_notes = batch_info["notes"]
        prompts = [build_llm_prompt(n) for n in batch_notes]

        try:
            outputs = generate_batch_json(
                prompts,
                max_new_tokens=batch_info["max_new_tokens"],
            )
        except Exception as e:
            failures.append({
                "bucket": batch_info["bucket"],
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })
            continue

        for note, raw_text in zip(batch_notes, outputs):
            raw_outputs.append({
                "note_id": note["note_id"],
                "page_num": note["page_num"],
                "raw_text": raw_text,
            })

            try:
                llm_mentions.extend(parse_and_clean_mentions(raw_text, note))
            except Exception as e:
                retry_raw = ""
                try:
                    retry_raw = generate_batch_json(
                        [build_llm_prompt(note)],
                        max_new_tokens=RETRY_MAX_NEW_TOKENS,
                    )[0]
                    raw_outputs.append({
                        "note_id": note["note_id"],
                        "page_num": note["page_num"],
                        "raw_text": retry_raw,
                        "retry": True,
                    })
                    llm_mentions.extend(parse_and_clean_mentions(retry_raw, note))
                except Exception as retry_e:
                    failures.append({
                        "note_id": note["note_id"],
                        "bucket": batch_info["bucket"],
                        "error": repr(e),
                        "retry_error": repr(retry_e),
                        "raw_text_preview": raw_text[:1200],
                        "retry_text_preview": retry_raw[:1200] if retry_raw else "",
                    })

    out = {
        "patient_code": patient_code,
        "source_pdf": doc["source_pdf"],
        "llm_mention_count": len(llm_mentions),
        "mentions": llm_mentions,
        "failures": failures,
    }
    save_json(out, out_path)

    if SAVE_RAW_LLM_OUTPUTS:
        save_json(
            {
                "patient_code": patient_code,
                "source_pdf": doc["source_pdf"],
                "raw_outputs": raw_outputs,
            },
            RAW_LLM_DIR / f"{patient_code}_raw_llm.json"
        )

    release_cuda()
    return out

llm_rows, llm_failures = [], []
selected_codes = set(selected_patient_codes())
unresolved_files = [
    fp for fp in sorted(UNRESOLVED_DIR.glob("*_unresolved.json"))
    if fp.stem.replace("_unresolved", "") in selected_codes
]
print("Unresolved files in current run:", len(unresolved_files))

for fp in tqdm(unresolved_files, desc="LLM rescue"):
    try:
        res = run_llm_on_unresolved_file(fp)
        llm_rows.append({
            "patient_code": res["patient_code"],
            "source_pdf": res["source_pdf"],
            "llm_mention_count": res["llm_mention_count"],
            "llm_failures": len(res["failures"]),
            "status": "ok",
        })
    except Exception as e:
        llm_failures.append({"file": fp.name, "error": repr(e), "traceback": traceback.format_exc()})
        logger.exception(f"LLM rescue failed: {fp.name}")

llm_df = pd.DataFrame(llm_rows)
llm_fail_df = pd.DataFrame(llm_failures)
llm_df.to_csv(REVIEW_DIR / "llm_inventory.csv", index=False)
llm_fail_df.to_csv(REVIEW_DIR / "llm_failures.csv", index=False)

display(llm_df.head(20))
display(llm_fail_df.head(10))


Unresolved files in current run: 147


LLM rescue:   0%|          | 0/147 [00:00<?, ?it/s]

batch=3 prompt_tokens=370 new=640 time=13.4s
batch=3 prompt_tokens=479 new=640 time=57.4s
batch=1 prompt_tokens=442 new=640 time=10.1s
batch=1 prompt_tokens=598 new=640 time=9.3s
batch=3 prompt_tokens=371 new=640 time=19.6s
batch=3 prompt_tokens=449 new=640 time=19.3s
batch=1 prompt_tokens=497 new=640 time=12.0s
batch=1 prompt_tokens=702 new=768 time=12.4s
batch=3 prompt_tokens=393 new=640 time=17.8s
batch=2 prompt_tokens=451 new=640 time=13.1s
batch=3 prompt_tokens=381 new=640 time=36.7s
batch=3 prompt_tokens=444 new=640 time=27.2s
batch=1 prompt_tokens=460 new=640 time=8.1s
batch=3 prompt_tokens=349 new=640 time=20.2s
batch=2 prompt_tokens=439 new=640 time=11.3s
batch=1 prompt_tokens=423 new=640 time=7.0s
batch=3 prompt_tokens=402 new=640 time=20.9s
batch=2 prompt_tokens=506 new=640 time=34.1s
batch=1 prompt_tokens=524 new=640 time=46.1s
batch=3 prompt_tokens=470 new=640 time=19.9s
batch=1 prompt_tokens=452 new=640 time=6.5s
batch=1 prompt_tokens=455 new=640 time=6.2s
batch=3 prompt_

,patient_code,source_pdf,llm_mention_count,llm_failures,status
0,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,2,0,ok
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,9,0,ok
2,ABDUL MABUD,ABDUL MABUD.pdf,9,0,ok
3,ABDUL RAJAK KAJI,ABDUL RAJAK KAJI.pdf,3,0,ok
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,17,0,ok
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,31,0,ok
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,43,0,ok
7,AMIR AL MORSHED,AMIR AL MORSHED.pdf,10,0,ok
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,15,0,ok
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,14,0,ok


""


In [ ]:
# Cell A — check disk usage under /home/pardeep only

from pathlib import Path
import os
import shutil
import subprocess

HOME_P = Path("/home/pardeep")
DATA_P = HOME_P / "data"
RUN_ROOT = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147")

print("=== Filesystem ===")
subprocess.run(["bash", "-lc", "df -h /home /home/pardeep /home/pardeep/data || true"])

print("\n=== Top directories under /home/pardeep (depth 2) ===")
subprocess.run([
    "bash", "-lc",
    r"du -xh --max-depth=2 /home/pardeep 2>/dev/null | sort -h | tail -n 50"
])

print("\n=== Run-root subdirectories ===")
if RUN_ROOT.exists():
    subprocess.run([
        "bash", "-lc",
        rf"du -xh --max-depth=2 '{RUN_ROOT}' 2>/dev/null | sort -h | tail -n 50"
    ])
else:
    print("RUN_ROOT not found:", RUN_ROOT)

=== Filesystem ===
The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Filesystem      Size  Used Avail Use% Mounted on
/dev/root       290G  290G     0 100% /
/dev/root       290G  290G     0 100% /
/dev/root       290G  290G     0 100% /

=== Top directories under /home/pardeep (depth 2) ===
8.0K	/home/pardeep/.conda
8.0K	/home/pardeep/.config/rclone
8.0K	/home/pardeep/.mamba
8.0K	/home/pardeep/.nv
8.0K	/home/pardeep/.ssh
8.0K	/home/pardeep/colab-runtime
12K	/home/pardeep/.cache/flashinfer
12K	/home/pardeep/.config/vllm
12K	/home/pardeep/.jupyter/lab
16K	/home/pardeep/.paddlex
20K	/home/pardeep/.jupyter
20K	/home/pardeep/.pyenv/completions
28K	/home/pardeep/.config
28K	/home/pardeep/.pyenv/man
36K	/home/pardeep/.cache/vllm
36K	/home/pardeep/.pyenv/src
40K	/home/pardeep/.cache/matplotlib
72K	/home/pardeep/.pyenv/pyenv.d
76K	/home/pardeep/.pyenv/shims
80K	/home/pardeep/.pyenv/.github
124K	/home/pa

In [ ]:
# Colab cell 1 — unload model/tokenizer from this kernel
import gc

for name in ["model", "tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass

print("Model/tokenizer cleared.")

Model/tokenizer cleared.


In [ ]:
# Colab cell 2 — remove only safe extras from current run
!find /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147 -type f -name "*.tmp" -delete
!find /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147 -type f -size 0 -delete
!rm -rf /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/hf_cache
!rm -rf /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/hf_offload
!df -h /home /home/pardeep /home/pardeep/data

Filesystem      Size  Used Avail Use% Mounted on
/dev/root       290G  287G  2.9G 100% /
/dev/root       290G  287G  2.9G 100% /
/dev/root       290G  287G  2.9G 100% /


In [ ]:
# Colab cell 4 — optional: clear only caches under /home/pardeep
!rm -rf /home/pardeep/.cache/huggingface
!rm -rf /home/pardeep/.cache/doctr
!rm -rf /home/pardeep/.cache/pip
!find /home/pardeep -type d -name "__pycache__" -prune -exec rm -rf {} +
!find /home/pardeep -type d -name ".ipynb_checkpoints" -prune -exec rm -rf {} +
!df -h /home /home/pardeep /home/pardeep/data

Filesystem      Size  Used Avail Use% Mounted on
/dev/root       290G  287G  3.6G  99% /
/dev/root       290G  287G  3.6G  99% /
/dev/root       290G  287G  3.6G  99% /


## This block focuses on trust and reviewability. Instead of letting an LLM rewrite the final patient story, the notebook merges mentions in Python. That is a very good design for medical extraction because it preserves provenance and reduces the risk of hallucinated clinical summaries. QC is also not treated as optional; the notebook writes review outputs so you can quickly see what succeeded, what failed, and what may need manual attention. Better alternatives could be a formal validation dashboard, a human review UI, or database-backed record comparison, but the notebook’s CSV/QC approach is lighter and suits a VM-first workflow. For this stage, simplicity is actually a strength.

In [ ]:
# Cell 12 — merge all mentions per patient without a lossy LLM rewrite

def flatten_mentions_for_patient(patient_code: str) -> List[Dict[str, Any]]:
    resolved_doc = load_json(RESOLVED_DIR / f"{patient_code}_resolved.json")
    llm_path = MENTION_DIR / f"{patient_code}_mentions.json"
    llm_doc = load_json(llm_path) if llm_path.exists() else {"mentions": []}
    mentions = []
    for note in resolved_doc["resolved_notes"]:
        mentions.extend(note.get("mentions", []))
    mentions.extend(llm_doc.get("mentions", []))
    return mentions

def merge_mentions(mentions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    merged = {}
    for m in mentions:
        if not isinstance(m, dict):
            continue
        category = normalize_text(m.get("category", "")) or "unknown"
        label = normalize_text(m.get("label", ""))
        value = normalize_text(m.get("normalized_value", "")) or normalize_text(m.get("value", ""))
        date_text = normalize_text(m.get("date_text", ""))
        key = (category.lower(), label.lower(), value.lower(), date_text.lower())

        cur = {
            "category": category,
            "label": label or category,
            "value": normalize_text(m.get("value", "")) or value,
            "normalized_value": value,
            "date_text": date_text,
            "certainty": str(m.get("certainty", "unknown")).lower(),
            "attributes": m.get("attributes", {}) if isinstance(m.get("attributes", {}), dict) else {},
            "source_pages": sorted(set(m.get("source_pages", []))),
            "evidence_ids": sorted(set(m.get("evidence_ids", []))),
            "evidence_quote": normalize_text(m.get("evidence_quote", "")),
            "origins": [m.get("origin", "unknown")],
        }

        if key not in merged:
            merged[key] = cur
        else:
            old = merged[key]
            old["source_pages"] = sorted(set(old["source_pages"]) | set(cur["source_pages"]))
            old["evidence_ids"] = sorted(set(old["evidence_ids"]) | set(cur["evidence_ids"]))
            old["origins"] = sorted(set(old["origins"]) | set(cur["origins"]))
            if not old["evidence_quote"] and cur["evidence_quote"]:
                old["evidence_quote"] = cur["evidence_quote"]
            if not old["attributes"] and cur["attributes"]:
                old["attributes"] = cur["attributes"]

    out = list(merged.values())
    out.sort(key=lambda x: (x["category"], x["date_text"], x["label"], x["normalized_value"]))
    return out

def group_mentions(mentions: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    groups = defaultdict(list)
    for m in mentions:
        groups[m["category"]].append(m)
    return dict(sorted(groups.items(), key=lambda x: x[0]))

def build_final_record(patient_code: str) -> Dict[str, Any]:
    note_doc = load_json(NOTE_LEDGER_DIR / f"{patient_code}_notes.json")
    page_doc = load_json(PAGE_LEDGER_DIR / f"{patient_code}_pages.json")
    all_mentions = flatten_mentions_for_patient(patient_code)
    merged_mentions = merge_mentions(all_mentions)
    grouped = group_mentions(merged_mentions)

    review_flags = []
    if not merged_mentions:
        review_flags.append("no_mentions_after_merge")
    if sum(1 for n in note_doc["notes"] if n["signal"] >= MIN_SIGNAL_FOR_LLM) > 0 and not merged_mentions:
        review_flags.append("clinical_signal_but_empty_output")

    return {
        "patient_code": patient_code,
        "source_pdf": note_doc["source_pdf"],
        "page_count": page_doc["page_count"],
        "note_count": note_doc["note_count"],
        "mentions": merged_mentions,
        "grouped_record": grouped,
        "review_flags": review_flags,
        "document_traceability": {
            "source_pages": sorted(set(p for m in merged_mentions for p in m.get("source_pages", []))),
            "evidence_ids": sorted(set(e for m in merged_mentions for e in m.get("evidence_ids", []))),
        },
        "stats": {
            "raw_mentions_before_merge": len(all_mentions),
            "mentions_after_merge": len(merged_mentions),
            "categories_after_merge": len(grouped),
        },
    }

final_rows, final_failures = [], []
patient_codes = selected_patient_codes()
print("Patients in current run:", len(patient_codes))

for patient_code in tqdm(patient_codes, desc="Final merge"):
    try:
        final = build_final_record(patient_code)
        save_json(final, FINAL_DIR / f"{patient_code}_final.json")
        final_rows.append({
            "patient_code": final["patient_code"],
            "source_pdf": final["source_pdf"],
            "mentions_after_merge": final["stats"]["mentions_after_merge"],
            "category_count": final["stats"]["categories_after_merge"],
            "review_flags": len(final["review_flags"]),
            "status": "ok",
        })
    except Exception as e:
        final_failures.append({"patient_code": patient_code, "error": repr(e), "traceback": traceback.format_exc()})
        logger.exception(f"Final merge failed: {patient_code}")

final_df = pd.DataFrame(final_rows)
final_fail_df = pd.DataFrame(final_failures)
final_df.to_csv(REVIEW_DIR / "final_inventory.csv", index=False)
final_fail_df.to_csv(REVIEW_DIR / "final_failures.csv", index=False)

display(final_df.head(20))
display(final_fail_df.head(10))


Patients in current run: 147


Final merge:   0%|          | 0/147 [00:00<?, ?it/s]

,patient_code,source_pdf,mentions_after_merge,category_count,review_flags,status
0,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,151,12,0,ok
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,382,11,0,ok
2,ABDUL MABUD,ABDUL MABUD.pdf,267,13,0,ok
3,ABDUL RAJAK KAJI,ABDUL RAJAK KAJI.pdf,145,12,0,ok
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,270,12,0,ok
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,574,13,0,ok
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,367,11,0,ok
7,AMIR AL MORSHED,AMIR AL MORSHED.pdf,118,10,0,ok
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,391,12,0,ok
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,504,15,0,ok


""


In [ ]:
# Cell 13 — QC summary

selected_codes = set(selected_patient_codes())
qc_rows = []

final_files = [
    fp for fp in sorted(FINAL_DIR.glob("*_final.json"))
    if fp.stem.replace("_final", "") in selected_codes
]

for fp in final_files:
    doc = load_json(fp)
    qc_rows.append({
        "patient_code": doc["patient_code"],
        "source_pdf": doc["source_pdf"],
        "mention_count": len(doc.get("mentions", [])),
        "category_count": len(doc.get("grouped_record", {})),
        "trace_pages": len(doc.get("document_traceability", {}).get("source_pages", [])),
        "trace_evidence_ids": len(doc.get("document_traceability", {}).get("evidence_ids", [])),
        "review_flags": len(doc.get("review_flags", [])),
        "raw_mentions_before_merge": doc.get("stats", {}).get("raw_mentions_before_merge", 0),
        "mentions_after_merge": doc.get("stats", {}).get("mentions_after_merge", 0),
    })

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(REVIEW_DIR / "qc_summary.csv", index=False)

display(qc_df.head(20))
if not qc_df.empty:
    display(qc_df.describe(include="all"))
    display(qc_df.sort_values("mention_count").head(15))
    display(qc_df.sort_values("mention_count", ascending=False).head(15))


,patient_code,source_pdf,mention_count,category_count,trace_pages,trace_evidence_ids,review_flags,raw_mentions_before_merge,mentions_after_merge
0,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,151,12,10,38,0,160,151
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,382,11,49,112,0,463,382
2,ABDUL MABUD,ABDUL MABUD.pdf,267,13,32,72,0,292,267
3,ABDUL RAJAK KAJI,ABDUL RAJAK KAJI.pdf,145,12,13,38,0,172,145
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,270,12,51,63,0,307,270
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,574,13,172,192,0,746,574
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,367,11,43,130,0,382,367
7,AMIR AL MORSHED,AMIR AL MORSHED.pdf,118,10,16,37,0,124,118
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,391,12,80,90,0,420,391
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,504,15,45,136,0,535,504


,patient_code,source_pdf,mention_count,category_count,trace_pages,trace_evidence_ids,review_flags,raw_mentions_before_merge,mentions_after_merge
count,147,147,147.000000,147.000000,147.000000,147.000000,147.0,147.000000,147.000000
unique,147,147,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,. RAMSHARAN SHRIVASTAVA,. RAMSHARAN SHRIVASTAVA.pdf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,118.074830,10.340136,16.476190,36.088435,0.0,126.258503,118.074830
std,NaN,NaN,92.698368,1.559314,17.585927,25.838785,0.0,106.415706,92.698368
min,NaN,NaN,14.000000,7.000000,2.000000,2.000000,0.0,14.000000,14.000000
25%,NaN,NaN,56.500000,9.000000,8.000000,19.500000,0.0,58.500000,56.500000
50%,NaN,NaN,94.000000,10.000000,12.000000,32.000000,0.0,96.000000,94.000000
75%,NaN,NaN,144.000000,11.500000,17.500000,42.000000,0.0,148.000000,144.000000


,patient_code,source_pdf,mention_count,category_count,trace_pages,trace_evidence_ids,review_flags,raw_mentions_before_merge,mentions_after_merge
111,Ritika Nayak,Ritika Nayak.pdf,14,8,2,2,0,14,14
75,Madhavankutty Panicker,Madhavankutty Panicker.pdf,16,7,3,4,0,16,16
63,MAHESH VIJAYVARGIYA,MAHESH VIJAYVARGIYA.pdf,17,8,7,7,0,23,17
40,Dilip Kumar Misra,Dilip Kumar Misra.pdf,22,8,7,9,0,22,22
39,Dhirendra Kumar Dey,Dhirendra Kumar Dey.pdf,29,9,7,13,0,29,29
34,Chiranjilal Jangid,Chiranjilal Jangid.pdf,31,9,5,11,0,33,31
141,VIRENDRA VIJAY NIVALKAR,VIRENDRA VIJAY NIVALKAR.pdf,31,7,6,15,0,32,31
61,Laxmi Wagh,Laxmi Wagh.pdf,32,10,5,11,0,35,32
133,Sujay Kumar Patra,Sujay Kumar Patra.pdf,32,7,6,12,0,32,32
74,MURUGAN ESSAKKIPPAN,MURUGAN ESSAKKIPPAN.pdf,34,7,4,17,0,34,34


,patient_code,source_pdf,mention_count,category_count,trace_pages,trace_evidence_ids,review_flags,raw_mentions_before_merge,mentions_after_merge
5,AJIT KUMAR CHOUDHARY,AJIT KUMAR CHOUDHARY.pdf,574,13,172,192,0,746,574
9,AMIT KUMAR PAL,AMIT KUMAR PAL.pdf,504,15,45,136,0,535,504
11,ANANTA CHETIA,ANANTA CHETIA.pdf,476,13,46,112,0,521,476
8,AMIT KUMAR MANDAL,AMIT KUMAR MANDAL.pdf,391,12,80,90,0,420,391
1,ABDUL GAFFAR SULEMAN,ABDUL GAFFAR SULEMAN.pdf,382,11,49,112,0,463,382
6,AKANKSHA TRIPATHI,AKANKSHA TRIPATHI.pdf,367,11,43,130,0,382,367
96,RAJA MOHAMMAD AMIR MOHAMMAD KHAN,RAJA MOHAMMAD AMIR MOHAMMAD KHAN.pdf,307,12,23,77,0,332,307
28,Babita Kumari,Babita Kumari.pdf,305,14,27,78,0,318,305
17,ASHIM SARMA,ASHIM SARMA.pdf,275,12,34,91,0,294,275
4,ADESH CHARANDAS YADAV,ADESH CHARANDAS YADAV.pdf,270,12,51,63,0,307,270


In [ ]:
# Cell 14 — inspect one final record

selected_codes = set(selected_patient_codes())
sample_files = [
    fp for fp in sorted(FINAL_DIR.glob("*_final.json"))
    if fp.stem.replace("_final", "") in selected_codes
]
print("Final files in current run:", len(sample_files))

if sample_files:
    sample = load_json(sample_files[0])
    print("Sample file:", sample_files[0].name)
    print("Patient:", sample["patient_code"])
    print("Mentions:", len(sample.get("mentions", [])))
    print("Categories:", list(sample.get("grouped_record", {}).keys())[:10])
    print(json.dumps(sample.get("mentions", [])[:8], indent=2, ensure_ascii=False))
else:
    print("No final files found.")


Final files in current run: 147
Sample file: . RAMSHARAN SHRIVASTAVA_final.json
Patient: . RAMSHARAN SHRIVASTAVA
Mentions: 151
Categories: ['diagnosis', 'follow_up', 'genomics', 'imaging', 'lab', 'medication', 'pathology', 'performance_status', 'plan', 'radiotherapy']
[
  {
    "category": "diagnosis",
    "label": "diagnosis",
    "value": "Right RCC recurrence",
    "normalized_value": "Right RCC recurrence",
    "date_text": "01/04/2022",
    "certainty": "confirmed",
    "attributes": {},
    "source_pages": [
      8
    ],
    "evidence_ids": [
      "p0008_n002"
    ],
    "evidence_quote": "Right RCC recurrence",
    "origins": [
      "regex"
    ]
  },
  {
    "category": "diagnosis",
    "label": "diagnosis",
    "value": "mRCC",
    "normalized_value": "mRCC",
    "date_text": "13/09/202",
    "certainty": "confirmed",
    "attributes": {},
    "source_pages": [
      2
    ],
    "evidence_ids": [
      "p0002_n008"
    ],
    "evidence_quote": "mRCC",
    "origins": [
   

This final block is about operational finish. Zipping outputs is necessary because the pipeline produces many intermediate and final folders, and you need a clean transfer unit after a long VM run. The dedicated full-run cell is also useful because it formalizes the workflow: validate on pilot first, then scale to the full dataset. Cleanup is especially important in notebook environments where GPU memory fragments easily after OCR plus LLM work. Better alternatives could be a scripted pipeline runner with automatic artifact versioning and post-run cleanup hooks, but the notebook’s explicit final cells are easier to understand and safer for manual execution.

In [ ]:
# Cell 15 — zip outputs for the current run

final_zip = shutil.make_archive(
    str(OUT_ROOT / "final_records_zip"),
    "zip",
    root_dir=str(OUT_ROOT),
    base_dir="final_records",
)
review_zip = shutil.make_archive(
    str(OUT_ROOT / "review_exports_zip"),
    "zip",
    root_dir=str(OUT_ROOT),
    base_dir="review_exports",
)
whole_run_zip = shutil.make_archive(
    str(OUT_ROOT),
    "zip",
    root_dir=str(OUT_ROOT.parent),
    base_dir=OUT_ROOT.name,
)

print("Created:", final_zip)
print("Created:", review_zip)
print("Created:", whole_run_zip)


Created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/final_records_zip.zip
Created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/review_exports_zip.zip
Created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147.zip


In [ ]:
from pathlib import Path
import os
import io
import re
import json
import zipfile
import traceback

import fitz
from PIL import Image, ImageChops

INPUT_PDF_DIR = Path("/home/pardeep/data/TMH_Patient_Reports")
BASE_OUT_ROOT = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized")
RUN_TAG = "full_147"   # matches your current full run folder
OUT_ROOT = BASE_OUT_ROOT / RUN_TAG

PROFILE_IMG_DIR = OUT_ROOT / "profile_images_only"
PROFILE_META_DIR = OUT_ROOT / "profile_images_only_meta"

PROFILE_IMG_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_META_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(INPUT_PDF_DIR.glob("*.pdf"))

print("INPUT_PDF_DIR:", INPUT_PDF_DIR)
print("OUT_ROOT:", OUT_ROOT)
print("PDF count:", len(pdf_files))
print("Images will be saved in:", PROFILE_IMG_DIR)

INPUT_PDF_DIR: /home/pardeep/data/TMH_Patient_Reports
OUT_ROOT: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147
PDF count: 147
Images will be saved in: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only


In [ ]:
def safe_stem(name: str) -> str:
    name = str(name or "").strip()
    name = re.sub(r"[^\w.\- ]+", "_", name)
    name = re.sub(r"\s+", "_", name).strip("._ ")
    return name or "unknown_pdf"

def trim_white_border(img: Image.Image) -> Image.Image:
    if img.mode != "RGB":
        img = img.convert("RGB")
    bg = Image.new("RGB", img.size, (255, 255, 255))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    if bbox:
        cropped = img.crop(bbox)
        if cropped.size[0] >= 20 and cropped.size[1] >= 20:
            return cropped
    return img

def save_png_from_bytes(image_bytes: bytes, out_path: Path):
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img = trim_white_border(img)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path, format="PNG")

def pick_patient_photo(page):
    """
    Prefer the small portrait-like image in the left-middle area
    below the TMH header. Tuned to your sample page layout.
    """
    try:
        infos = page.get_image_info(xrefs=True)
    except Exception:
        infos = []

    pw = float(page.rect.width)
    ph = float(page.rect.height)
    candidates = []

    for info in infos:
        xref = info.get("xref", 0)
        bbox = info.get("bbox")
        if not xref or not bbox:
            continue

        rect = fitz.Rect(bbox)
        bw = float(rect.width)
        bh = float(rect.height)
        if bw <= 0 or bh <= 0:
            continue

        area_ratio = (bw * bh) / max(pw * ph, 1.0)
        aspect = bw / max(bh, 1.0)

        # avoid top logos/header images
        if rect.y0 < 0.12 * ph:
            continue

        # patient pic is usually on left side
        if rect.x0 > 0.35 * pw:
            continue

        # patient pic is usually small portrait-ish
        if not (0.0008 <= area_ratio <= 0.05):
            continue
        if not (0.35 <= aspect <= 1.35):
            continue

        score = 0.0
        if rect.x0 <= 0.25 * pw:
            score += 3
        if 0.18 * ph <= rect.y0 <= 0.50 * ph:
            score += 3
        if bh >= bw:
            score += 2
        score += min(area_ratio * 100, 2)

        candidates.append({
            "score": score,
            "xref": xref,
            "bbox": [round(rect.x0, 2), round(rect.y0, 2), round(rect.x1, 2), round(rect.y1, 2)],
        })

    if not candidates:
        return None

    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates[0]

def crop_photo_fallback(page, out_path: Path):
    """
    Fallback crop based on your sample PDF layout.
    """
    r = page.rect

    clip = fitz.Rect(
        r.x0 + 0.06 * r.width,
        r.y0 + 0.24 * r.height,
        r.x0 + 0.25 * r.width,
        r.y0 + 0.43 * r.height,
    )

    pix = page.get_pixmap(clip=clip, dpi=220, alpha=False)
    tmp_path = out_path.with_suffix(".tmp.png")
    pix.save(str(tmp_path))

    img = Image.open(tmp_path).convert("RGB")
    img = trim_white_border(img)
    img.save(out_path, format="PNG")

    try:
        tmp_path.unlink(missing_ok=True)
    except Exception:
        pass

    return [round(clip.x0, 2), round(clip.y0, 2), round(clip.x1, 2), round(clip.y1, 2)]

def extract_one_pdf_photo(pdf_path: Path):
    patient_code = safe_stem(pdf_path.stem)
    out_img = PROFILE_IMG_DIR / f"{patient_code}.png"
    out_meta = PROFILE_META_DIR / f"{patient_code}.json"

    meta = {
        "patient_code": patient_code,
        "source_pdf": pdf_path.name,
        "photo_found": False,
        "photo_file": "",
        "method": "",
        "bbox": None,
        "error": "",
    }

    try:
        doc = fitz.open(pdf_path)
        page = doc[0]

        picked = pick_patient_photo(page)
        if picked is not None:
            try:
                img_info = doc.extract_image(picked["xref"])
                image_bytes = img_info.get("image")
                if image_bytes:
                    save_png_from_bytes(image_bytes, out_img)
                    if out_img.exists() and out_img.stat().st_size > 0:
                        meta["photo_found"] = True
                        meta["photo_file"] = out_img.name
                        meta["method"] = "embedded_image"
                        meta["bbox"] = picked["bbox"]
            except Exception:
                pass

        if not meta["photo_found"]:
            bbox = crop_photo_fallback(page, out_img)
            if out_img.exists() and out_img.stat().st_size > 0:
                meta["photo_found"] = True
                meta["photo_file"] = out_img.name
                meta["method"] = "page_crop_fallback"
                meta["bbox"] = bbox

        doc.close()

    except Exception as e:
        meta["error"] = repr(e)

    out_meta.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta

In [ ]:
rows = []
failures = []

for i, pdf_path in enumerate(pdf_files, 1):
    meta = extract_one_pdf_photo(pdf_path)
    rows.append(meta)

    if not meta["photo_found"]:
        failures.append(meta)

    if i % 10 == 0 or i == len(pdf_files):
        print(f"done {i}/{len(pdf_files)}")

print("Total PDFs:", len(rows))
print("Photos extracted:", sum(1 for r in rows if r["photo_found"]))
print("Failures:", len(failures))

done 10/147
done 20/147
done 30/147
done 40/147
done 50/147
done 60/147
done 70/147
done 80/147
done 90/147
done 100/147
done 110/147
done 120/147
done 130/147
done 140/147
done 147/147
Total PDFs: 147
Photos extracted: 147
Failures: 0


In [ ]:
zip_path = OUT_ROOT / "profile_images_only_zip.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for img_path in sorted(PROFILE_IMG_DIR.glob("*.png")):
        zf.write(img_path, arcname=img_path.name)

print("Created:", zip_path)
print("Image folder:", PROFILE_IMG_DIR)
print("Image count:", len(list(PROFILE_IMG_DIR.glob('*.png'))))

Created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only_zip.zip
Image folder: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only
Image count: 147


In [ ]:
from pathlib import Path
from shutil import copy2
from IPython.display import HTML, display

ZIP_PATH = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only_zip.zip")
assert ZIP_PATH.exists(), f"Missing file: {ZIP_PATH}"

local_name = Path.cwd() / ZIP_PATH.name
copy2(ZIP_PATH, local_name)

display(HTML(f'<a href="{local_name.name}" download>Download profile_images_only_zip.zip</a>'))

In [ ]:
from pathlib import Path
from shutil import copy2
from IPython.display import HTML, display

ZIP_PATH = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only_zip.zip")
assert ZIP_PATH.exists(), f"Missing file: {ZIP_PATH}"

local_name = Path.cwd() / ZIP_PATH.name
copy2(ZIP_PATH, local_name)

display(HTML(f'<a href="{local_name.name}" download>Download profile_images_only_zip.zip</a>'))

In [ ]:
from pathlib import Path
import zipfile

PROFILE_IMG_DIR = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only")
ZIP_PATH = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only_zip.zip")

assert PROFILE_IMG_DIR.exists(), f"Folder not found: {PROFILE_IMG_DIR}"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for img_path in sorted(PROFILE_IMG_DIR.glob("*.png")):
        zf.write(img_path, arcname=img_path.name)

print("Created:", ZIP_PATH)
print("Total images zipped:", len(list(PROFILE_IMG_DIR.glob('*.png'))))

Created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147/profile_images_only_zip.zip
Total images zipped: 147


In [ ]:
# Cell 15 — zip outputs for the current run and download directly from Colab

from pathlib import Path
import zipfile
import os

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

RUN_ROOT = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147")
ZIP_PATH = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147_outputs_bundle.zip")

# Keep only useful outputs; skip model/cache/temp folders
INCLUDE_DIRS = [
    "page_ledger",
    "note_ledger",
    "resolved_notes",
    "unresolved_notes",
    "mentions",
    "final_records",
    "review_exports",
    "logs",
]

INCLUDE_FILES = [
    "run_manifest.json",
]

EXCLUDE_SUFFIXES = {".tmp"}
EXCLUDE_NAMES = {
    "hf_cache",
    "hf_offload",
    "tmp_images",
    "raw_llm_outputs",
    ".ipynb_checkpoints",
    "__pycache__",
}

if not RUN_ROOT.exists():
    raise FileNotFoundError(f"Run folder not found: {RUN_ROOT}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

added = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    # Add selected folders
    for dname in INCLUDE_DIRS:
        dpath = RUN_ROOT / dname
        if not dpath.exists():
            continue

        for fp in dpath.rglob("*"):
            if not fp.is_file():
                continue
            if fp.suffix in EXCLUDE_SUFFIXES:
                continue
            if any(part in EXCLUDE_NAMES for part in fp.parts):
                continue
            rel = fp.relative_to(RUN_ROOT)
            zf.write(fp, arcname=str(rel))
            added += 1

    # Add selected top-level files
    for fname in INCLUDE_FILES:
        fpath = RUN_ROOT / fname
        if fpath.exists() and fpath.is_file():
            zf.write(fpath, arcname=fpath.name)
            added += 1

zip_size_mb = ZIP_PATH.stat().st_size / (1024**2)
print(f"ZIP created: {ZIP_PATH}")
print(f"Files added: {added}")
print(f"ZIP size: {zip_size_mb:.2f} MB")

if IN_COLAB:
    files.download(str(ZIP_PATH))
else:
    print("Not running in Colab frontend, so direct browser download was skipped.")
    print("Download manually from:", ZIP_PATH)

ZIP created: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147_outputs_bundle.zip
Files added: 893
ZIP size: 6.27 MB
Not running in Colab frontend, so direct browser download was skipped.
Download manually from: /home/pardeep/data/Evolet_Qwen15B_Optimized/full_147_outputs_bundle.zip


In [ ]:
from pathlib import Path
import shutil

src = Path("/home/pardeep/data/Evolet_Qwen15B_Optimized/full_147_outputs_bundle.zip")
dst = Path("/home/pardeep/colab-runtime/full_147_outputs_bundle.zip")

shutil.copy2(src, dst)
print("Ready:")
print("http://localhost:8892/files/full_147_outputs_bundle.zip?download=1")

Ready:
http://localhost:8892/files/full_147_outputs_bundle.zip?download=1
